# 🎬 Video2X trên Kaggle (AI Video Upscaling & Frame Interpolation)

Notebook này được tối ưu để chạy **[Video2X v6.2.0](https://github.com/k4yt3x/video2x)** (phiên bản C++ native hiệu năng cao) trên nền tảng **Kaggle** sử dụng GPU miễn phí (**NVIDIA Tesla T4** hoặc **P100**).

---
### ⚠️ Các cài đặt bắt buộc trên Kaggle trước khi chạy:
1. **Bật GPU:** Ở bảng **Notebook Settings** bên phải màn hình ➡️ mục **Accelerator** ➡️ chọn **GPU T4 x2** (hoặc GPU P100).
2. **Bật Internet:** Ở bảng **Notebook Settings** ➡️ gạt toggle **Internet** sang **ON** (bắt buộc để tải Video2X, FFmpeg 7 và Vulkan libraries).
3. **Quản lý file:**
   - Video sau khi xử lý sẽ lưu tại `/kaggle/working/output/`.
   - Bạn có thể tải video về máy trực tiếp từ tab **Output** ở panel bên phải hoặc bấm vào link tải ngay trong notebook.


## 🖥️ Bước 0: Kiểm tra phần cứng GPU
Chạy lệnh sau để xác nhận Kaggle đã cấp phát GPU NVIDIA và driver đang hoạt động.


In [ ]:
# Kiểm tra thông tin GPU được cấp phát
!nvidia-smi


## 📦 Bước 1: Cài đặt Video2X v6.2.0, FFmpeg 7 & Kích hoạt Vulkan NVIDIA

### Các thành phần cài đặt:
1. **FFmpeg 7 (`libavcodec.so.61`)**: Bắt buộc vì Video2X v6.2.0 được biên dịch với FFmpeg 7.
2. **NVIDIA Vulkan Driver (`libnvidia-gl-*`)**: Cài đặt thư viện đồ họa Vulkan của NVIDIA vào container để sửa lỗi `Failed to create Vulkan instance`.
3. **Video2X v6.2.0**: Bản binary C++ native tốc độ cao.


In [ ]:
import os
import re
import json
import subprocess
from pathlib import Path

print("⏳ [1/5] Thêm PPA FFmpeg 7 và cập nhật danh sách gói...")
!apt-get update -yqq
!apt-get install -yqq software-properties-common
!add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg7
!apt-get update -yqq
!apt-get install -yqq --no-install-recommends libvulkan1 libvulkan-dev vulkan-tools ffmpeg mesa-vulkan-drivers

print("\n⏳ [2/5] Nhận diện và cài đặt thư viện Vulkan NVIDIA (libnvidia-gl)...")
# Tự động nhận diện phiên bản driver NVIDIA từ nvidia-smi
driver_ver = '535'
try:
    smi_out = subprocess.check_output('nvidia-smi', shell=True).decode()
    m = re.search(r'Driver Version:\s*(\d+)', smi_out)
    if m:
        driver_ver = m.group(1)
        print(f'🔍 Phiên bản driver phát hiện: {driver_ver}')
except Exception as e:
    print(f'⚠️ Không đọc được nvidia-smi, sử dụng mặc định 535: {e}')

# Cài đặt thư viện đồ họa user-space NVIDIA Vulkan (sửa lỗi Failed to create Vulkan instance)
ret = os.system(f'apt-get install -yqq --no-install-recommends libnvidia-gl-{driver_ver}')
if ret != 0:
    print(f'⚠️ libnvidia-gl-{driver_ver} không có sẵn, thử cài libnvidia-gl-535...')
    os.system('apt-get install -yqq --no-install-recommends libnvidia-gl-535')

print("\n⏳ [3/5] Tải và cài đặt Video2X v6.2.0...")
deb_file = "/tmp/video2x-linux-ubuntu2204-amd64.deb"
!wget -q --show-progress -O {deb_file} https://github.com/k4yt3x/video2x/releases/download/6.2.0/video2x-linux-ubuntu2204-amd64.deb
!apt-get install -yqq {deb_file} || (dpkg -i {deb_file} && apt-get install -fyqq)

print("\n⏳ [4/5] Cấu hình Vulkan ICD cho NVIDIA GPU...")
icd_dir = Path("/usr/share/vulkan/icd.d")
icd_dir.mkdir(parents=True, exist_ok=True)
nvidia_icd_path = icd_dir / "nvidia_icd.json"

# Tạo file ICD trỏ đến thư viện NVIDIA
if not nvidia_icd_path.exists():
    with open(nvidia_icd_path, "w") as f:
        json.dump({
            "file_format_version": "1.0.0",
            "ICD": {
                "library_path": "libGLX_nvidia.so.0",
                "api_version": "1.3.0"
            }
        }, f, indent=4)

!cp -f /usr/share/vulkan/icd.d/nvidia_icd.json /etc/vulkan/icd.d/ 2>/dev/null || true

# Thiết lập biến môi trường
os.environ["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/nvidia_icd.json"
os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__VK_LAYER_NV_optimus"] = "NVIDIA_only"
!ldconfig

print("\n⏳ [5/5] Kiểm tra thiết bị Vulkan và Video2X...")
!vulkaninfo --summary 2>/dev/null | grep -E "deviceName|driverName" || !vulkaninfo --summary || true
!video2x --version

print("\n✅ Cài đặt hoàn tất! GPU NVIDIA đã sẵn sàng cho Video2X.")


## 📥 Bước 2: Tải Video từ thiết bị lên Kaggle

Bấm nút **"📤 Tải video lên"** bên dưới để chọn file video từ máy tính của bạn (`.mp4`, `.mkv`, `.avi`, `.mov`, `.webm`,...).
File sẽ tự động được tải lên Kaggle và sẵn sàng cho quá trình xử lý ở Bước 3.


In [ ]:
import os
import pathlib
import ipywidgets as widgets
from IPython.display import display, HTML

WORKING_INPUT = pathlib.Path("/kaggle/working/input")
WORKING_INPUT.mkdir(parents=True, exist_ok=True)

selected_input_file = None

# Khung hiển thị trạng thái video
status_box = widgets.HTML(value="""
<div style='background-color: #fef9e7; padding: 12px; border-radius: 8px; border-left: 5px solid #f39c12; margin-top: 10px;'>
    <b style='color:#d35400;'>⚠️ Hãy bấm nút bên dưới để chọn file video từ thiết bị của bạn.</b>
</div>
""")

# Nút tải video trực tiếp từ máy tính
uploader = widgets.FileUpload(
    accept=".mp4,.mkv,.avi,.mov,.webm,.flv,.ts,.m4v",
    multiple=False,
    description="📤 Tải video lên",
    button_style="primary",
    layout=widgets.Layout(width="220px", height="42px")
)

upload_output = widgets.Output()

def on_file_uploaded(change):
    global selected_input_file
    with upload_output:
        upload_output.clear_output()
        if not uploader.value:
            return
        try:
            uploaded_data = uploader.value
            fname = None
            content = None
            
            # Hỗ trợ tương thích cả ipywidgets v7 và v8
            if isinstance(uploaded_data, dict):
                for name, item in uploaded_data.items():
                    fname = name
                    content = item['content']
                    break
            elif isinstance(uploaded_data, (list, tuple)):
                for item in uploaded_data:
                    fname = item.get('name', 'uploaded_video.mp4') if isinstance(item, dict) else getattr(item, 'name', 'uploaded_video.mp4')
                    content = item.get('content', b'') if isinstance(item, dict) else getattr(item, 'content', b'')
                    break
            
            if fname and content:
                dest = WORKING_INPUT / fname
                with open(dest, "wb") as f:
                    f.write(content)
                selected_input_file = dest
                size_mb = len(content) / (1024 * 1024)
                status_box.value = f"""
                <div style='background-color: #eafaf1; padding: 14px; border-radius: 8px; border-left: 5px solid #2ecc71; margin-top: 10px;'>
                    <b style='color:#27ae60; font-size:16px;'>🎉 Tải lên thành công! Video đã sẵn sàng:</b><br>
                    🎬 <b>Tên video:</b> {dest.name}<br>
                    📁 <b>Đường dẫn:</b> <code>{dest}</code><br>
                    ⚖️ <b>Dung lượng:</b> <b>{size_mb:.2f} MB</b><br><br>
                    👉 <i>Bạn có thể chuyển sang Bước 3 bên dưới để bắt đầu Upscale!</i>
                </div>
                """
        except Exception as e:
            status_box.value = f"""
            <div style='background-color: #fdedec; padding: 12px; border-radius: 8px; border-left: 5px solid #e74c3c; margin-top: 10px;'>
                <b style='color:#c0392b;'>❌ Lỗi khi tải file lên:</b> {e}
            </div>
            """

uploader.observe(on_file_uploaded, names='value')

# Tự động phát hiện nếu đã có video tải lên trước đó
existing_files = sorted(WORKING_INPUT.glob("*"), key=os.path.getmtime, reverse=True)
if existing_files and selected_input_file is None:
    selected_input_file = existing_files[0]
    size_mb = selected_input_file.stat().st_size / (1024 * 1024)
    status_box.value = f"""
    <div style='background-color: #eafaf1; padding: 14px; border-radius: 8px; border-left: 5px solid #2ecc71; margin-top: 10px;'>
        <b style='color:#27ae60; font-size:16px;'>🎯 Đang sử dụng video đã tải lên:</b><br>
        🎬 <b>Tên video:</b> {selected_input_file.name}<br>
        📁 <b>Đường dẫn:</b> <code>{selected_input_file}</code><br>
        ⚖️ <b>Dung lượng:</b> <b>{size_mb:.2f} MB</b><br><br>
        👉 <i>(Hoặc bấm nút bên trên nếu bạn muốn tải lên video khác)</i>
    </div>
    """

display(uploader)
display(upload_output)
display(status_box)


## ⚙️ Bước 3: Cấu hình và Thực thi Video Upscale

### 💡 Hướng dẫn chọn tham số:
* **`PROCESSOR`**:
  * `realesrgan`: Nâng độ phân giải bằng Deep Learning với chất lượng vượt trội.
  * `libplacebo`: Dùng bộ shader Anime4K GLSL, tốc độ cực kỳ nhanh cho anime/hoạt hình.
* **`REALESRGAN_MODEL`**:
  * `realesr-animevideov3`: Chuyên dụng cho hoạt hình/anime (hỗ trợ scale 2x, 3x, 4x) - **Khuyên dùng**.
  * `realesrgan-plus-anime`: Chất lượng cực cao cho anime (chỉ hỗ trợ scale 4x).
  * `realesrgan-plus`: Tối ưu cho video đời thực, người thật, phong cảnh (chỉ hỗ trợ 4x).
* **`CRF` (Chất lượng video xuất)**: Giá trị từ 18 (rất nét, file to) đến 24 (dung lượng nhẹ hơn). Mặc định `20` là mức tối ưu.


In [ ]:
import os
import time
import pathlib
import subprocess

# ==========================================================
# ⚙️ TÙY CHỈNH THÔNG SỐ UPSCALING TẠI ĐÂY
# ==========================================================
PROCESSOR = "realesrgan"            # "realesrgan" hoặc "libplacebo"

# Cấu hình Real-ESRGAN
REALESRGAN_MODEL = "realesr-animevideov3"  # "realesr-animevideov3", "realesrgan-plus-anime", "realesrgan-plus"
SCALE = 4                                  # 2, 3, hoặc 4 (Model animevideov3 hỗ trợ 2, 3, 4; model khác chỉ 4)

# Cấu hình libplacebo (Anime4K)
SHADER = "anime4k-v4-a"             # "anime4k-v4-a", "anime4k-v4-b", "anime4k-v4-c", "anime4k-v4.1-gan"
TARGET_WIDTH = 1920                 # Đặt 0 nếu muốn tự tính toán
TARGET_HEIGHT = 1080                # Đặt 0 nếu muốn tự tính toán

# Cấu hình FFmpeg Encoder
CODEC = "libx264"                   # "libx264" (CPU x264 chất lượng cao) hoặc "h264_nvenc" (NVIDIA Hardware)
PRESET = "slow"                    # "ultrafast", "fast", "medium", "slow", "veryslow"
CRF = 20                            # 17 - 23 (số càng thấp chất lượng càng cao)
PIXFMT = "auto"                     # "auto", "yuv420p"
BITRATE = 0                         # 0 để dùng CRF
LOGLEVEL = "info"                   # "info", "debug", "warn", "error"

# Thư mục lưu kết quả
OUTPUT_DIR = pathlib.Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ==========================================================

# Kiểm tra đầu vào
if 'selected_input_file' not in globals() or selected_input_file is None or not pathlib.Path(selected_input_file).exists():
    raise ValueError("❌ Bạn chưa chọn video đầu vào! Vui lòng hoàn thành Bước 2 trước.")

input_path = pathlib.Path(selected_input_file)
output_path = OUTPUT_DIR / f"{input_path.stem}_upscaled{input_path.suffix}"

# Xóa file cũ nếu đã tồn tại để tránh xung đột
if output_path.exists():
    output_path.unlink()

print(f"🎬 Video nguồn: {input_path}")
print(f"💾 Video đích  : {output_path}")
print(f"🚀 Processor  : {PROCESSOR} | Scale: {SCALE}x")

# Xây dựng lệnh video2x
cmd = [
    "video2x",
    "--input", f'"{input_path}"',
    "--output", f'"{output_path}"',
    "--processor", PROCESSOR,
    "--codec", CODEC,
    "--log-level", LOGLEVEL,
    "-e", f"preset={PRESET}",
    "-e", f"crf={CRF}"
]

if BITRATE > 0:
    cmd.extend(["--bit-rate", str(BITRATE)])

if PIXFMT != "auto":
    cmd.extend(["--pix-fmt", PIXFMT])

if PROCESSOR == "realesrgan":
    cmd.extend([
        "--realesrgan-model", REALESRGAN_MODEL,
        "--scaling-factor", str(SCALE)
    ])
elif PROCESSOR == "libplacebo":
    cmd.extend(["--libplacebo-shader", f'"{SHADER}"'])
    if TARGET_WIDTH > 0:
        cmd.extend(["--width", str(TARGET_WIDTH)])
    if TARGET_HEIGHT > 0:
        cmd.extend(["--height", str(TARGET_HEIGHT)])

full_cmd = " ".join(cmd)
print("-" * 60)
print(f"Executing: {full_cmd}")
print("-" * 60)

start_time = time.time()
p = subprocess.Popen(full_cmd, shell=True)
p.communicate()
elapsed = time.time() - start_time

# Kiểm tra kết quả (Lưu ý: Video2X có thể thoát mã 139 lúc giải phóng Vulkan nhưng file đã ghi xong)
if output_path.exists() and output_path.stat().st_size > 1024:
    size_mb = output_path.stat().st_size / (1024 * 1024)
    print("=" * 60)
    if p.returncode in (0, 139, -11):
        print(f"🎉 XỬ LÝ THÀNH CÔNG trong {elapsed:.1f} giây!")
        if p.returncode in (139, -11):
            print("ℹ️ Ghi chú: Mã thoát 139 (SIGSEGV) xuất hiện khi hủy Vulkan driver lúc đóng ứng dụng, video đã hoàn tất 100%.")
    else:
        print(f"⚠️ Video đã xuất ra ({size_mb:.2f} MB), nhưng chương trình trả về mã thoát: {p.returncode}")
    print(f"📁 File kết quả: {output_path} ({size_mb:.2f} MB)")
    print("=" * 60)
else:
    print("=" * 60)
    print(f"❌ Đã xảy ra lỗi khi xử lý! (Exit code: {p.returncode})")
    print("=" * 60)


## 📺 Bước 4: Xem trước (Preview) & Tải Video
Sau khi xử lý xong, bạn có thể xem trước video trực tiếp ngay trên notebook và nhấp vào liên kết tải xuống.


In [ ]:
from IPython.display import display, HTML, FileLink
import pathlib

if 'output_path' in globals() and output_path.exists():
    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"✅ Video sẵn sàng tải về: {output_path.name} ({size_mb:.2f} MB)\n")
    
    # 1. Hiển thị link tải trực tiếp
    print("👇 Bấm vào liên kết bên dưới để tải file về máy:")
    display(FileLink(str(output_path)))
    
    # 2. Preview video player nếu là file MP4
    if output_path.suffix.lower() == ".mp4":
        display(HTML(f"""
        <div style="margin-top: 15px;">
            <h4>🎬 Xem trước video:</h4>
            <video width="640" height="360" controls autoplay muted>
                <source src="{output_path}" type="video/mp4">
                Trình duyệt của bạn không hỗ trợ thẻ video HTML5.
            </video>
        </div>
        """))
else:
    print("⚠️ Chưa tìm thấy file video kết quả. Hãy chạy Bước 3 trước.")


## ⏩ Bước 5 (Tùy chọn): Tăng FPS với mô hình RIFE
Video2X hỗ trợ mô hình **RIFE (Real-Time Intermediate Flow Estimation)** để nhân số khung hình (ví dụ nhân đôi từ 24fps/30fps lên 60fps), giúp chuyển động mượt mà hơn đáng kể.


In [ ]:
import os
import time
import pathlib
import subprocess

# Cấu hình RIFE
RIFE_MODEL = "rife-v4.6"       # "rife-v4.6", "rife-anime", "rife-v4", "rife-UHD", "rife-v2.4"
FRAME_RATE_MUL = 2             # Hệ số nhân FPS (2: nhân đôi, 4: nhân 4)
CODEC = "libx264"
PRESET = "slow"
CRF = 20

OUTPUT_DIR = pathlib.Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if 'selected_input_file' not in globals() or not pathlib.Path(selected_input_file).exists():
    raise ValueError("❌ Chưa chọn video đầu vào!")

input_path = pathlib.Path(selected_input_file)
output_rife = OUTPUT_DIR / f"{input_path.stem}_rife_{FRAME_RATE_MUL}x{input_path.suffix}"

if output_rife.exists():
    output_rife.unlink()

rife_cmd = [
    "video2x",
    "--input", f'"{input_path}"',
    "--output", f'"{output_rife}"',
    "--processor", "rife",
    "--rife-model", RIFE_MODEL,
    "--frame-rate-mul", str(FRAME_RATE_MUL),
    "--codec", CODEC,
    "--log-level", "info",
    "-e", f"preset={PRESET}",
    "-e", f"crf={CRF}"
]

full_rife_cmd = " ".join(rife_cmd)
print(f"🚀 Đang chạy RIFE frame interpolation:\n{full_rife_cmd}\n" + "-"*60)

start_t = time.time()
res = os.system(full_rife_cmd)
dur = time.time() - start_t

# Kiểm tra kết quả
if output_rife.exists() and output_rife.stat().st_size > 1024:
    mb = output_rife.stat().st_size / (1024 * 1024)
    print("=" * 60)
    print(f"🎉 Hoàn tất RIFE trong {dur:.1f} giây!")
    print(f"📁 File kết quả: {output_rife} ({mb:.2f} MB)")
    print("=" * 60)
else:
    print("=" * 60)
    print("⚠️ RIFE không hoàn thành thành công (kiểm tra lại log chi tiết ở trên).")
    print("=" * 60)
